# The Sandpit: Understanding Optimization with Jacobians

## Introduction

Welcome to the **optimization playground**! In this notebook, we'll explore how **Jacobians** help us find maxima and minima of functions, and understand the challenges of optimization in real-world scenarios.

We'll use the metaphor of a **sandpit** - imagine trying to find the deepest point in a sandpit using only a long stick, without being able to see the bottom. This captures the essence of many optimization problems where we can only evaluate the function at specific points.

## Learning Objectives

By the end of this notebook, you will understand:
- How **Jacobians point toward steeper slopes** (gradients)
- The difference between **local** and **global** extrema
- Why **optimization can be challenging** in practice
- The **sandpit analogy** for real-world optimization problems
- How **gradient-based methods** work and their limitations

## Key Concepts

**Optimization**: Finding input values that make a function as large (maximum) or small (minimum) as possible.

**Jacobian/Gradient**: Points in the direction of steepest increase, like road signs saying "peak this way!"

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as patches

# Set up plotting parameters
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully!")
print("Ready to explore the optimization sandpit!")

## Part 1: Simple Optimization - The Easy Case

### Starting with a Simple Function

Let's begin with the simplest case mentioned in the transcript. Consider the function:

$$f(x, y) = e^{-(x^2 + y^2)}$$

This is our **Gaussian hill** - it has a single, clear maximum at the origin.

### Finding the Maximum Analytically

For simple functions, we can find extrema by:

1. **Computing the Jacobian (gradient)**:
   $$\nabla f = \begin{bmatrix} \frac{\partial f}{\partial x} \\ \frac{\partial f}{\partial y} \end{bmatrix}$$

2. **Setting it equal to zero**:
   $$\nabla f = \mathbf{0}$$

3. **Solving for the critical points**

### Why This Works
At extrema (maxima, minima, or saddle points), the gradient is zero - there's no direction of steepest ascent because you're at a "flat" point.

In [ ]:
# Define the simple Gaussian function
def simple_function(x, y):
    """Simple Gaussian hill: f(x,y) = exp(-(x² + y²))"""
    return np.exp(-(x**2 + y**2))

def simple_jacobian(x, y):
    """Jacobian of the simple function"""
    f_val = simple_function(x, y)
    return np.array([-2*x*f_val, -2*y*f_val])

# Solve analytically for the maximum
print("Analytical Solution for Simple Function")
print("="*45)
print("Function: f(x,y) = exp(-(x² + y²))")
print()
print("Step 1: Compute the Jacobian")
print("∂f/∂x = -2x · exp(-(x² + y²))")
print("∂f/∂y = -2y · exp(-(x² + y²))")
print()
print("Step 2: Set Jacobian = 0")
print("[-2x · exp(-(x² + y²)), -2y · exp(-(x² + y²))] = [0, 0]")
print()
print("Step 3: Solve")
print("Since exp(-(x² + y²)) > 0 for all (x,y):")
print("-2x · exp(-(x² + y²)) = 0  ⟹  x = 0")
print("-2y · exp(-(x² + y²)) = 0  ⟹  y = 0")
print()
print("Solution: Maximum at (0, 0)")
print(f"Maximum value: f(0,0) = {simple_function(0, 0):.4f}")

# Verify this is indeed a maximum by checking nearby points
test_points = [(0.1, 0), (0, 0.1), (-0.1, 0), (0, -0.1)]
print(f"\nVerification (nearby points should have smaller values):")
for x, y in test_points:
    val = simple_function(x, y)
    print(f"f({x:4.1f}, {y:4.1f}) = {val:.6f}")

print(f"\nAll nearby values < {simple_function(0, 0):.4f} ✓ Confirmed maximum!")

In [ ]:
# Visualize the simple optimization problem
fig = plt.figure(figsize=(16, 12))

# Create meshgrid for plotting
x = np.linspace(-3, 3, 100)
y = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x, y)
Z = simple_function(X, Y)

# Coarser grid for gradient arrows
x_arrows = np.linspace(-3, 3, 15)
y_arrows = np.linspace(-3, 3, 15)
X_arrows, Y_arrows = np.meshgrid(x_arrows, y_arrows)

# Calculate gradient field
JX = np.zeros_like(X_arrows)
JY = np.zeros_like(Y_arrows)
for i in range(X_arrows.shape[0]):
    for j in range(X_arrows.shape[1]):
        grad = simple_jacobian(X_arrows[i,j], Y_arrows[i,j])
        JX[i,j] = grad[0]
        JY[i,j] = grad[1]

# Plot 1: 3D Surface
ax1 = fig.add_subplot(2, 2, 1, projection='3d')
surf = ax1.plot_surface(X, Y, Z, cmap='viridis', alpha=0.8)
ax1.scatter([0], [0], [1], color='red', s=100, label='Maximum')
ax1.set_title('3D Surface: Simple Gaussian Hill')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_zlabel('f(x,y)')
ax1.legend()

# Plot 2: Contour plot with gradient field
ax2 = fig.add_subplot(2, 2, 2)
contour = ax2.contourf(X, Y, Z, levels=20, cmap='viridis')
plt.colorbar(contour, ax=ax2, label='f(x,y)')

# Add gradient arrows (pointing uphill)
ax2.quiver(X_arrows, Y_arrows, JX, JY, 
          alpha=0.7, scale=20, color='white', width=0.003)

# Mark the maximum
ax2.plot(0, 0, 'ro', markersize=10, label='Maximum at (0,0)')
ax2.set_title('Contour Plot with Gradient Field\n(Arrows point uphill)')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.legend()

# Plot 3: Cross-section along x-axis
ax3 = fig.add_subplot(2, 2, 3)
x_line = np.linspace(-3, 3, 100)
z_line = simple_function(x_line, 0)
ax3.plot(x_line, z_line, 'b-', linewidth=2, label='f(x,0)')
ax3.axvline(x=0, color='red', linestyle='--', alpha=0.7, label='Maximum')
ax3.scatter([0], [1], color='red', s=100, zorder=5)
ax3.set_title('Cross-section along x-axis')
ax3.set_xlabel('x')
ax3.set_ylabel('f(x,0)')
ax3.grid(True, alpha=0.3)
ax3.legend()

# Plot 4: "Road signs" illustration
ax4 = fig.add_subplot(2, 2, 4)
ax4.contour(X, Y, Z, levels=10, colors='gray', alpha=0.5)

# Show several points with their gradient vectors
sample_points = [(-2, -1), (-1, 1), (1, -1), (2, 1), (1.5, 0)]
for px, py in sample_points:
    grad = simple_jacobian(px, py)
    # Scale gradient for visibility
    grad_scaled = grad * 0.8
    
    ax4.plot(px, py, 'bo', markersize=8)
    ax4.arrow(px, py, grad_scaled[0], grad_scaled[1], 
             head_width=0.1, head_length=0.1, fc='red', ec='red')
    
    # Add "road sign" text
    ax4.annotate('Peak\nthis way!', 
                xy=(px + grad_scaled[0], py + grad_scaled[1]),
                xytext=(px + grad_scaled[0] + 0.3, py + grad_scaled[1] + 0.2),
                fontsize=8, ha='center',
                bbox=dict(boxstyle="round,pad=0.2", facecolor="yellow", alpha=0.7),
                arrowprops=dict(arrowstyle='->', color='red'))

ax4.plot(0, 0, 'ro', markersize=12, label='Peak (all signs point here)')
ax4.set_title('"Road Signs" Point Toward Peak')
ax4.set_xlabel('x')
ax4.set_ylabel('y')
ax4.legend()
ax4.set_xlim(-3, 3)
ax4.set_ylim(-3, 3)

plt.tight_layout()
plt.show()

print("Key Observations:")
print("• The gradient arrows all point toward the maximum at (0,0)")
print("• At the maximum, the gradient is zero (no arrows)")
print("• This is why we set ∇f = 0 to find extrema")
print("• The 'road signs' analogy: gradients point toward peaks")

## Part 2: The Challenge - Multiple Extrema

### When Things Get Complicated

Real-world functions are rarely as simple as our Gaussian hill. Consider this more complex function with **multiple peaks and valleys**:

$$f(x, y) = 3e^{-x^2-y^2} - 2e^{-(x-1.5)^2-(y-1)^2} + e^{-(x+1)^2-(y+1)^2} - 0.5e^{-(x-1)^2-(y+2)^2} + 0.8e^{-(x+2)^2-y^2}$$

### The Problems This Creates

1. **Multiple solutions** when we set $\nabla f = 0$
2. **Local vs global extrema** - many peaks, but which is tallest?
3. **Analytical solutions become impossible** for complex functions
4. **We need new strategies** beyond just solving $\nabla f = 0$

### Types of Extrema

- **Global Maximum**: The highest peak overall (point A in transcript)  
- **Local Maxima**: High points that are peaks in their neighborhood (points C, E)  
- **Global Minimum**: The lowest valley overall (point D)  
- **Local Minima**: Low points in their local area (point B)  

### The Key Challenge
**How do we find the global optimum when we can't see the whole landscape?**

In [ ]:
# Define a complex function with multiple extrema
def complex_function(x, y):
    """Complex function with multiple maxima and minima"""
    term1 = 3 * np.exp(-x**2 - y**2)                              # Main peak at origin
    term2 = -2 * np.exp(-(x-1.5)**2 - (y-1)**2)                 # Valley 
    term3 = np.exp(-(x+1)**2 - (y+1)**2)                        # Secondary peak
    term4 = -0.5 * np.exp(-(x-1)**2 - (y+2)**2)                 # Small valley
    term5 = 0.8 * np.exp(-(x+2)**2 - y**2)                      # Small peak
    return term1 + term2 + term3 + term4 + term5

def complex_jacobian(x, y):
    """Numerical gradient of the complex function"""
    h = 1e-7
    df_dx = (complex_function(x + h, y) - complex_function(x - h, y)) / (2 * h)
    df_dy = (complex_function(x, y + h) - complex_function(x, y - h)) / (2 * h)
    return np.array([df_dx, df_dy])

# Create the complex landscape
x = np.linspace(-3, 3, 200)
y = np.linspace(-3, 3, 200)
X, Y = np.meshgrid(x, y)
Z_complex = complex_function(X, Y)

# Find approximate locations of extrema by checking grid points
def find_extrema(X, Y, Z, threshold=0.1):
    """Find approximate locations of local extrema"""
    maxima = []
    minima = []
    
    # Check interior points
    for i in range(1, Z.shape[0]-1):
        for j in range(1, Z.shape[1]-1):
            center = Z[i, j]
            neighbors = Z[i-1:i+2, j-1:j+2]
            
            # Check if it's a local maximum
            if center == np.max(neighbors) and center > np.mean(neighbors) + threshold:
                maxima.append((X[i, j], Y[i, j], center))
            
            # Check if it's a local minimum  
            elif center == np.min(neighbors) and center < np.mean(neighbors) - threshold:
                minima.append((X[i, j], Y[i, j], center))
    
    return maxima, minima

maxima, minima = find_extrema(X, Y, Z_complex)

print("Complex Function Analysis")
print("="*35)
print(f"Found {len(maxima)} local maxima:")
for i, (x_max, y_max, z_max) in enumerate(maxima):
    print(f"  Maximum {chr(65+i)}: ({x_max:.2f}, {y_max:.2f}) → f = {z_max:.3f}")

print(f"\nFound {len(minima)} local minima:")
for i, (x_min, y_min, z_min) in enumerate(minima):
    print(f"  Minimum {chr(65+i)}: ({x_min:.2f}, {y_min:.2f}) → f = {z_min:.3f}")

# Identify global extrema
if maxima:
    global_max = max(maxima, key=lambda x: x[2])
    print(f"\n🏔️  GLOBAL MAXIMUM: ({global_max[0]:.2f}, {global_max[1]:.2f}) → f = {global_max[2]:.3f}")

if minima:
    global_min = min(minima, key=lambda x: x[2])
    print(f"🕳️  GLOBAL MINIMUM: ({global_min[0]:.2f}, {global_min[1]:.2f}) → f = {global_min[2]:.3f}")

print(f"\nThe Challenge:")
print(f"• Setting ∇f = 0 gives {len(maxima) + len(minima)} different solutions!")
print(f"• How do we know which one is the global optimum?")
print(f"• What if we can only evaluate f at a few points?")

In [ ]:
# Visualize the complex landscape
fig = plt.figure(figsize=(18, 12))

# Plot 1: 3D Surface showing the complexity
ax1 = fig.add_subplot(2, 3, 1, projection='3d')
surf = ax1.plot_surface(X, Y, Z_complex, cmap='terrain', alpha=0.8)

# Mark all extrema on 3D plot
for i, (x_max, y_max, z_max) in enumerate(maxima):
    ax1.scatter([x_max], [y_max], [z_max], color='red', s=100, 
               label=f'Max {chr(65+i)}' if i < 3 else '')

for i, (x_min, y_min, z_min) in enumerate(minima):
    ax1.scatter([x_min], [y_min], [z_min], color='blue', s=100,
               label=f'Min {chr(65+i)}' if i < 3 else '')

ax1.set_title('Complex 3D Landscape')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_zlabel('f(x,y)')
if len(maxima) > 0 or len(minima) > 0:
    ax1.legend()

# Plot 2: Contour plot - the "map view"
ax2 = fig.add_subplot(2, 3, 2)
contour = ax2.contourf(X, Y, Z_complex, levels=25, cmap='terrain')
plt.colorbar(contour, ax=ax2, label='f(x,y)')

# Mark extrema on contour plot
for i, (x_max, y_max, z_max) in enumerate(maxima):
    ax2.plot(x_max, y_max, 'r^', markersize=12, markeredgecolor='black',
            label=f'Maximum {chr(65+i)}')
    ax2.annotate(f'{chr(65+i)}', (x_max, y_max), xytext=(5, 5), 
                textcoords='offset points', fontweight='bold')

for i, (x_min, y_min, z_min) in enumerate(minima):
    ax2.plot(x_min, y_min, 'bv', markersize=12, markeredgecolor='black',
            label=f'Minimum {chr(65+i)}')
    ax2.annotate(f'{chr(65+i)}', (x_min, y_min), xytext=(5, -15), 
                textcoords='offset points', fontweight='bold')

ax2.set_title('Contour Map - "What We See in Daylight"')
ax2.set_xlabel('x')
ax2.set_ylabel('y')

# Plot 3: Gradient field showing the "road signs"
ax3 = fig.add_subplot(2, 3, 3)
ax3.contour(X, Y, Z_complex, levels=15, colors='gray', alpha=0.4)

# Sample gradient at various points
x_grad = np.linspace(-3, 3, 12)
y_grad = np.linspace(-3, 3, 12)
X_grad, Y_grad = np.meshgrid(x_grad, y_grad)

# Calculate gradients
GX = np.zeros_like(X_grad)
GY = np.zeros_like(Y_grad)
for i in range(X_grad.shape[0]):
    for j in range(X_grad.shape[1]):
        grad = complex_jacobian(X_grad[i,j], Y_grad[i,j])
        GX[i,j] = grad[0]
        GY[i,j] = grad[1]

# Plot gradient arrows
ax3.quiver(X_grad, Y_grad, GX, GY, alpha=0.7, scale=50, color='purple', width=0.004)

# Mark extrema
for x_max, y_max, z_max in maxima:
    ax3.plot(x_max, y_max, 'r^', markersize=10)

ax3.set_title('Gradient Field - "Road Signs"')
ax3.set_xlabel('x')
ax3.set_ylabel('y')

# Plot 4: The "night walking" scenario
ax4 = fig.add_subplot(2, 3, 4)
ax4.set_facecolor('black')
ax4.contour(X, Y, Z_complex, levels=15, colors='darkgray', alpha=0.3)

# Show a few "torch-lit" areas with gradients
torch_points = [(-1.5, -0.5), (0.5, 1.5), (1.8, -1.2)]
for tx, ty in torch_points:
    # Create a "torch light" circle
    circle = patches.Circle((tx, ty), 0.4, facecolor='yellow', alpha=0.3, edgecolor='orange')
    ax4.add_patch(circle)
    
    # Show gradient in this lit area
    grad = complex_jacobian(tx, ty)
    if np.linalg.norm(grad) > 0.01:  # Only show if gradient is significant
        ax4.arrow(tx, ty, grad[0]*0.5, grad[1]*0.5, head_width=0.1, head_length=0.1, 
                 fc='red', ec='red', linewidth=2)
        ax4.text(tx, ty-0.6, 'Peak\nthis way!', ha='center', va='top', 
                color='white', fontsize=8, fontweight='bold')

ax4.set_title('Night Walking - "Can Only See Locally"')
ax4.set_xlabel('x')
ax4.set_ylabel('y')
ax4.set_xlim(-3, 3)
ax4.set_ylim(-3, 3)

# Plot 5: Cross-section showing multiple peaks
ax5 = fig.add_subplot(2, 3, 5)
x_cross = np.linspace(-3, 3, 300)
y_cross = 0  # Cross-section along y=0
z_cross = complex_function(x_cross, y_cross)

ax5.plot(x_cross, z_cross, 'b-', linewidth=2, label='f(x, 0)')

# Mark peaks and valleys in cross-section
for x_max, y_max, z_max in maxima:
    if abs(y_max - y_cross) < 0.3:  # Near the cross-section line
        ax5.axvline(x=x_max, color='red', linestyle='--', alpha=0.7)
        ax5.plot(x_max, z_max, 'r^', markersize=8)

for x_min, y_min, z_min in minima:
    if abs(y_min - y_cross) < 0.3:
        ax5.axvline(x=x_min, color='blue', linestyle='--', alpha=0.7)
        ax5.plot(x_min, z_min, 'bv', markersize=8)

ax5.set_title('Cross-section: Multiple Peaks and Valleys')
ax5.set_xlabel('x')
ax5.set_ylabel('f(x, 0)')
ax5.grid(True, alpha=0.3)

# Plot 6: The challenge illustration
ax6 = fig.add_subplot(2, 3, 6)
ax6.text(0.5, 0.8, "The Optimization Challenge", ha='center', va='center', 
         fontsize=16, fontweight='bold', transform=ax6.transAxes)

challenges = [
    "🔍 Multiple peaks and valleys",
    "❓ Which is the global optimum?", 
    "🌙 Can't see the whole landscape",
    "💰 Function evaluation is expensive",
    "🎯 Gradients point to nearest peak",
    "⚠️  Easy to get trapped in local optima"
]

for i, challenge in enumerate(challenges):
    ax6.text(0.1, 0.65 - i*0.1, challenge, ha='left', va='center', 
             fontsize=12, transform=ax6.transAxes)

ax6.set_xlim(0, 1)
ax6.set_ylim(0, 1)
ax6.axis('off')

plt.tight_layout()
plt.show()

print("\nKey Insights from Complex Function:")
print("• Gradients still point uphill, but to the NEAREST peak")
print("• Following gradients might lead to local maximum, not global")
print("• In 'night walking', you could get stuck at a local peak")
print("• Need better strategies for global optimization")

## Part 3: The Sandpit Analogy

### Why Switch from Hill-Walking to Sandpit?

The **hill-walking analogy** has some misleading features for real optimization:

❌ **Misleading aspects of hill-walking:**
- Suggests you have to "walk" continuously between points
- Implies distance matters for computational cost
- Makes it seem like you can see nearby terrain

✅ **Why the sandpit is better:**
- **Discrete sampling**: You can "teleport" to any point to test it
- **Equal cost**: Each function evaluation costs the same regardless of location
- **No visibility**: You can't see the shape - just probe with your stick
- **Realistic constraints**: Matches real optimization scenarios

### The Sandpit Setup

Imagine a **very deep sandpit** with an uneven bottom:
- 🏖️ **Sand blocks your view** - you can't see the bottom shape
- 📏 **Long stick for probing** - measures depth at any point you choose
- 🚫 **No sideways movement** - once the stick is in, you can't move it around
- 📍 **Discrete measurements** - pull out stick, choose new location, repeat

### Real-World Parallels

This matches many optimization scenarios:
- **Engineering design**: Each design requires expensive simulation/testing
- **Drug discovery**: Each compound requires costly laboratory experiments  
- **Machine learning**: Each hyperparameter set requires full model training
- **Business strategy**: Each strategy requires market testing/implementation

In [ ]:
# Sandpit Exploration Simulation
class SandpitExplorer:
    def __init__(self, function, bounds=(-3, 3)):
        self.function = function
        self.bounds = bounds
        self.measurements = []
        self.num_probes = 0
        
    def probe(self, x, y):
        """Probe the sandpit at position (x, y)"""
        if not (self.bounds[0] <= x <= self.bounds[1] and self.bounds[0] <= y <= self.bounds[1]):
            print(f"❌ Can't probe outside sandpit! Bounds: [{self.bounds[0]}, {self.bounds[1]}]")
            return None
            
        depth = -self.function(x, y)  # Negative because we're measuring depth
        self.measurements.append((x, y, depth))
        self.num_probes += 1
        
        print(f"🏖️ Probe #{self.num_probes}: Position ({x:.2f}, {y:.2f}) → Depth: {depth:.3f}")
        return depth
    
    def estimate_gradient(self, x, y, delta=0.1):
        """Estimate gradient using nearby measurements"""
        # Probe nearby points
        depths = {}
        for dx, dy in [(delta, 0), (-delta, 0), (0, delta), (0, -delta)]:
            nx, ny = x + dx, y + dy
            if self.bounds[0] <= nx <= self.bounds[1] and self.bounds[0] <= ny <= self.bounds[1]:
                depths[(dx, dy)] = -self.function(nx, ny)
        
        # Estimate partial derivatives
        if (delta, 0) in depths and (-delta, 0) in depths:
            df_dx = (depths[(delta, 0)] - depths[(-delta, 0)]) / (2 * delta)
        else:
            df_dx = 0
            
        if (0, delta) in depths and (0, -delta) in depths:
            df_dy = (depths[(0, delta)] - depths[(0, -delta)]) / (2 * delta)
        else:
            df_dy = 0
            
        return np.array([df_dx, df_dy])
    
    def show_exploration_map(self):
        """Show what we've discovered so far"""
        if len(self.measurements) == 0:
            print("🤷 No measurements yet! Try probing some points.")
            return
            
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # True landscape (what we can't see)
        x = np.linspace(self.bounds[0], self.bounds[1], 100)
        y = np.linspace(self.bounds[0], self.bounds[1], 100)
        X, Y = np.meshgrid(x, y)
        Z = -self.function(X, Y)  # Negative for depth
        
        ax1.contourf(X, Y, Z, levels=20, cmap='terrain')
        ax1.set_title('🚫 Hidden Landscape\n(What we CANNOT see)')
        ax1.set_xlabel('x position')
        ax1.set_ylabel('y position')
        
        # Our measurements (what we can see)
        ax2.set_facecolor('tan')  # Sand color
        
        if len(self.measurements) > 0:
            xs, ys, depths = zip(*self.measurements)
            scatter = ax2.scatter(xs, ys, c=depths, cmap='terrain', s=100, 
                                edgecolors='black', linewidth=2)
            plt.colorbar(scatter, ax=ax2, label='Measured Depth')
            
            # Number the points in order
            for i, (x, y, d) in enumerate(self.measurements):
                ax2.annotate(f'{i+1}', (x, y), xytext=(0, 0), textcoords='offset points',
                           ha='center', va='center', fontweight='bold', color='white')
        
        ax2.set_title('🏖️ Our Measurements\n(What we KNOW)')
        ax2.set_xlabel('x position')
        ax2.set_ylabel('y position')
        ax2.set_xlim(self.bounds[0], self.bounds[1])
        ax2.set_ylim(self.bounds[0], self.bounds[1])
        
        plt.tight_layout()
        plt.show()
        
        # Show best guess so far
        if len(self.measurements) > 0:
            best_x, best_y, best_depth = max(self.measurements, key=lambda x: x[2])
            print(f"\n🎯 Current best guess for deepest point:")
            print(f"   Position: ({best_x:.2f}, {best_y:.2f})")
            print(f"   Depth: {best_depth:.3f}")
            print(f"   Based on {self.num_probes} probes")

# Create sandpit explorer with our complex function
sandpit = SandpitExplorer(complex_function)

print("🏖️ Welcome to the Sandpit Optimization Challenge!")
print("="*55)
print("Your mission: Find the DEEPEST point in the sandpit")
print("Tools: A long stick to measure depth at any location")
print("Constraints: You can't see the bottom shape - only probe!")
print()
print("Available methods:")
print("• sandpit.probe(x, y) - Measure depth at position (x,y)")  
print("• sandpit.show_exploration_map() - See your measurements")
print("• sandpit.estimate_gradient(x, y) - Estimate local slope")
print()
print("Let's start with some example probes...")

# Demonstrate with a few probes
example_probes = [(0, 0), (-1, -1), (1.5, 1), (-2, 0.5)]
for x, y in example_probes:
    sandpit.probe(x, y)

print()
sandpit.show_exploration_map()

## Part 4: Optimization Strategies and Challenges

### The Core Challenges

From our sandpit exploration, we can identify key optimization challenges:

1. **Limited Information**: We can only sample a few points
2. **Local vs Global**: Gradients lead to nearest optimum, not necessarily best
3. **Expensive Evaluations**: Each probe/measurement has a cost
4. **No Global View**: Can't see the entire landscape at once

### Common Optimization Strategies

#### 1. **Gradient Descent** 🔄
- Follow gradients uphill (or downhill for minimization)
- **Pros**: Efficient for smooth, convex functions
- **Cons**: Gets trapped in local optima

#### 2. **Random Search** 🎲  
- Sample points randomly across the space
- **Pros**: Can escape local optima
- **Cons**: Inefficient, ignores gradient information

#### 3. **Grid Search** 📊
- Systematically sample on a regular grid
- **Pros**: Thorough coverage
- **Cons**: Expensive, curse of dimensionality

#### 4. **Multi-Start Methods** 🔄🔄🔄
- Run gradient descent from multiple random starting points
- **Pros**: Better chance of finding global optimum
- **Cons**: Computationally expensive

### The "Nighttime Walking" Problem

Remember the analogy: walking at night with only local information (torch/gradient signs):

- ✅ **Gradients show direction of steepest ascent**
- ❌ **But only point to nearest peak, not highest peak**
- ⚠️ **Easy to get stuck at local maximum**
- 🔦 **Limited "visibility" of the landscape**

In [ ]:
# Demonstrate different optimization strategies
def gradient_ascent_step(x, y, function, step_size=0.1):
    """Single step of gradient ascent"""
    # Estimate gradient numerically
    h = 1e-6
    grad_x = (function(x + h, y) - function(x - h, y)) / (2 * h)
    grad_y = (function(x, y + h) - function(x, y - h)) / (2 * h)
    
    # Take step in gradient direction
    new_x = x + step_size * grad_x
    new_y = y + step_size * grad_y
    
    return new_x, new_y, np.array([grad_x, grad_y])

def run_gradient_ascent(start_x, start_y, function, max_steps=50, step_size=0.1, tolerance=1e-6):
    """Run gradient ascent from a starting point"""
    path = [(start_x, start_y, function(start_x, start_y))]
    x, y = start_x, start_y
    
    for step in range(max_steps):
        new_x, new_y, gradient = gradient_ascent_step(x, y, function, step_size)
        
        # Check bounds
        new_x = np.clip(new_x, -3, 3)
        new_y = np.clip(new_y, -3, 3)
        
        new_value = function(new_x, new_y)
        path.append((new_x, new_y, new_value))
        
        # Check for convergence
        if np.linalg.norm(gradient) < tolerance:
            break
            
        x, y = new_x, new_y
    
    return path

# Visualize different optimization strategies
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Create base landscape
x = np.linspace(-3, 3, 100)
y = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x, y)
Z = complex_function(X, Y)

# Strategy 1: Single Gradient Ascent (can get stuck)
ax1 = axes[0, 0]
ax1.contourf(X, Y, Z, levels=20, cmap='terrain', alpha=0.7)

# Run gradient ascent from one starting point
start_point = (-2, -2)
path1 = run_gradient_ascent(start_point[0], start_point[1], complex_function)

if len(path1) > 1:
    xs, ys, values = zip(*path1)
    ax1.plot(xs, ys, 'r-', linewidth=3, alpha=0.8, label='Gradient ascent path')
    ax1.plot(xs[0], ys[0], 'go', markersize=10, label='Start')
    ax1.plot(xs[-1], ys[-1], 'ro', markersize=10, label='End (local max)')

ax1.set_title('Strategy 1: Single Gradient Ascent\n(Can get trapped!)')
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.legend()

# Strategy 2: Multi-start Gradient Ascent
ax2 = axes[0, 1]
ax2.contourf(X, Y, Z, levels=20, cmap='terrain', alpha=0.7)

# Multiple starting points
start_points = [(-2, -2), (2, 2), (-1, 1), (1, -1), (0, 2)]
colors = ['red', 'blue', 'green', 'orange', 'purple']

best_value = -np.inf
best_point = None

for i, (sx, sy) in enumerate(start_points):
    path = run_gradient_ascent(sx, sy, complex_function)
    if len(path) > 1:
        xs, ys, values = zip(*path)
        ax2.plot(xs, ys, color=colors[i], linewidth=2, alpha=0.7, label=f'Path {i+1}')
        ax2.plot(xs[0], ys[0], 'o', color=colors[i], markersize=8)
        ax2.plot(xs[-1], ys[-1], 's', color=colors[i], markersize=8)
        
        if values[-1] > best_value:
            best_value = values[-1]
            best_point = (xs[-1], ys[-1])

if best_point:
    ax2.plot(best_point[0], best_point[1], 'k*', markersize=15, label='Best found')

ax2.set_title('Strategy 2: Multi-start Gradient Ascent\n(Better chance of global optimum)')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.legend()

# Strategy 3: Random Search
ax3 = axes[1, 0]
ax3.contourf(X, Y, Z, levels=20, cmap='terrain', alpha=0.7)

# Random sampling
np.random.seed(42)
n_random = 50
random_x = np.random.uniform(-3, 3, n_random)
random_y = np.random.uniform(-3, 3, n_random)
random_values = [complex_function(x, y) for x, y in zip(random_x, random_y)]

# Color points by their values
scatter = ax3.scatter(random_x, random_y, c=random_values, cmap='viridis', 
                     s=50, edgecolors='black', linewidth=1)
plt.colorbar(scatter, ax=ax3, label='Function value')

# Mark the best random point
best_idx = np.argmax(random_values)
ax3.plot(random_x[best_idx], random_y[best_idx], 'r*', markersize=15, 
         label=f'Best random: {random_values[best_idx]:.3f}')

ax3.set_title('Strategy 3: Random Search\n(Can escape local optima)')
ax3.set_xlabel('x')
ax3.set_ylabel('y')
ax3.legend()

# Strategy 4: Grid Search
ax4 = axes[1, 1]
ax4.contourf(X, Y, Z, levels=20, cmap='terrain', alpha=0.7)

# Grid search
n_grid = 10
grid_x = np.linspace(-3, 3, n_grid)
grid_y = np.linspace(-3, 3, n_grid)
Grid_X, Grid_Y = np.meshgrid(grid_x, grid_y)
grid_values = complex_function(Grid_X, Grid_Y)

# Plot grid points
scatter = ax4.scatter(Grid_X, Grid_Y, c=grid_values, cmap='viridis', 
                     s=50, marker='s', edgecolors='black', linewidth=1)
plt.colorbar(scatter, ax=ax4, label='Function value')

# Mark the best grid point
best_idx = np.unravel_index(np.argmax(grid_values), grid_values.shape)
best_grid_x = Grid_X[best_idx]
best_grid_y = Grid_Y[best_idx]
best_grid_value = grid_values[best_idx]

ax4.plot(best_grid_x, best_grid_y, 'r*', markersize=15, 
         label=f'Best grid: {best_grid_value:.3f}')

ax4.set_title('Strategy 4: Grid Search\n(Systematic but expensive)')
ax4.set_xlabel('x')
ax4.set_ylabel('y')
ax4.legend()

plt.tight_layout()
plt.show()

# Compare strategy effectiveness
print("Strategy Comparison")
print("="*40)
print(f"Single gradient ascent: Found value {path1[-1][2]:.3f}")
print(f"Multi-start gradient:   Found value {best_value:.3f}")
print(f"Random search:         Found value {max(random_values):.3f}")
print(f"Grid search:           Found value {best_grid_value:.3f}")
print()
print("Key Insights:")
print("• Single gradient ascent gets trapped in local optimum")
print("• Multi-start is better but requires more function evaluations")
print("• Random search can find good points but is inefficient")
print("• Grid search is systematic but expensive for high dimensions")

## Summary: The Optimization Playground

### What We've Learned

✅ **Jacobians as "Road Signs"** - Gradients point toward steeper slopes  
✅ **Simple vs Complex Functions** - Easy analytical solutions vs multiple extrema  
✅ **Local vs Global Optima** - The difference between nearest peak and highest peak  
✅ **The Sandpit Analogy** - Why discrete sampling better represents real problems  
✅ **Optimization Strategies** - Different approaches with various trade-offs  

### Key Insights

🎯 **Optimization is about finding the best input values for a function**  
🗺️ **Jacobians provide local directional information (gradients)**  
🌙 **"Night walking" captures the challenge of limited information**  
🏖️ **Sandpit analogy better represents discrete, expensive evaluations**  
🔄 **Multiple strategies needed for complex landscapes**  

### Real-World Applications

**Engineering Design** 🔧
- Optimizing aircraft wing shapes (expensive wind tunnel tests)
- Designing efficient engines (complex simulations)
- Circuit design optimization

**Machine Learning** 🤖  
- Hyperparameter tuning (expensive training runs)
- Neural architecture search
- Finding optimal model parameters

**Business & Finance** 💼
- Portfolio optimization (market testing required)
- Supply chain optimization  
- Resource allocation problems

**Scientific Research** 🔬
- Drug discovery (costly lab experiments)
- Materials science (expensive synthesis)
- Climate modeling parameter tuning

### The Fundamental Trade-offs

**Exploration vs Exploitation**
- Explore new areas vs exploit known good regions
- Random search vs gradient-based methods
- Global search vs local refinement

**Accuracy vs Efficiency**  
- More function evaluations → better results → higher cost
- Simple methods vs sophisticated algorithms
- Stopping criteria and convergence tolerance

### Questions for Further Exploration

1. How do we balance exploration and exploitation?
2. What if the function is noisy or has measurement errors?
3. How do we handle constraints (boundaries, feasibility)?
4. What about high-dimensional spaces (curse of dimensionality)?
5. How do we know when we've found a "good enough" solution?

The sandpit has given us a foundation for understanding these deeper optimization challenges!

In [ ]:
# Encourage further exploration
print("🏖️ Continue Your Sandpit Adventure!")
print("="*45)
print()
print("The sandpit explorer is still available for you to experiment with:")
print()
print("Try these challenges:")
print("1. Can you find the global minimum using only 10 probes?")
print("2. What happens if you start from different corners?")
print("3. Can you develop a systematic search strategy?")
print("4. How would you combine gradient information with exploration?")
print()
print("Example commands to continue exploring:")
print("• sandpit.probe(1.2, -0.8)")
print("• sandpit.estimate_gradient(0, 0)")  
print("• sandpit.show_exploration_map()")
print()
print("Remember: In real optimization, every function evaluation has a cost!")
print("The art is finding the best solution with the fewest evaluations.")
print()

# Create a final visualization showing the journey
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

# Show the complete landscape one more time
x = np.linspace(-3, 3, 100)
y = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x, y)
Z = complex_function(X, Y)

contour = ax.contourf(X, Y, Z, levels=25, cmap='terrain', alpha=0.8)
plt.colorbar(contour, ax=ax, label='Function Value')

# Add title and labels
ax.set_title('The Optimization Journey: From Simple Hills to Complex Landscapes', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('x')
ax.set_ylabel('y')

# Add annotations for key concepts
ax.annotate('Simple Case:\nSingle peak\nEasy to solve analytically', 
           xy=(0, 0), xytext=(-2.5, 2.5),
           arrowprops=dict(arrowstyle='->', color='blue', lw=2),
           bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue", alpha=0.8),
           fontsize=10)

ax.annotate('Complex Reality:\nMultiple extrema\nRequires smart strategies', 
           xy=(1.5, 1), xytext=(2, -2),
           arrowprops=dict(arrowstyle='->', color='red', lw=2),
           bbox=dict(boxstyle="round,pad=0.3", facecolor="lightcoral", alpha=0.8),
           fontsize=10)

ax.annotate('The Challenge:\nFind global optimum\nwith limited information', 
           xy=(-1, -1), xytext=(-2.8, -1.5),
           arrowprops=dict(arrowstyle='->', color='purple', lw=2),
           bbox=dict(boxstyle="round,pad=0.3", facecolor="plum", alpha=0.8),
           fontsize=10)

plt.tight_layout()
plt.show()

print("🎓 Congratulations!")
print("You've completed the journey from simple optimization to understanding")
print("the real-world challenges of finding optimal solutions in complex landscapes!")
print()
print("Next steps: Apply these concepts to real optimization problems in")
print("machine learning, engineering, and scientific research!")